In [1]:
!pip uninstall -y onnx onnxruntime neural-compressor
!pip install "onnx==1.14.1" "onnxruntime==1.17.1" "neural-compressor==2.5" onnx-simplifier

Found existing installation: onnx 1.14.1
Uninstalling onnx-1.14.1:
  Successfully uninstalled onnx-1.14.1
Found existing installation: onnxruntime 1.17.1
Uninstalling onnxruntime-1.17.1:
  Successfully uninstalled onnxruntime-1.17.1
Found existing installation: neural_compressor 2.5
Uninstalling neural_compressor-2.5:
  Successfully uninstalled neural_compressor-2.5
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 117.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 119.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 129.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import os
import numpy as np
import onnx
from onnx import numpy_helper
import onnxruntime
from onnxsim import simplify
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat, quant_pre_process
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat
from onnxruntime.quantization import CalibrationMethod
from PIL import Image
from torchvision import transforms
from neural_compressor import quantization, PostTrainingQuantConfig
from neural_compressor.data import DataLoader
from neural_compressor.metric import Metric
import shutil

from neural_compressor import quantization
from neural_compressor.config import PostTrainingQuantConfig, AccuracyCriterion, TuningCriterion
from neural_compressor.data import DataLoader
from neural_compressor.metric import Metric
import shutil

/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py:644: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


NOTE! Installing ujson may make loading annotations faster.


In [3]:

BASE_DIR = os.getcwd()
INPUT_MODEL_PATH = os.path.join(BASE_DIR, "weights", "yolop-640-640.onnx")
OUTPUT_DIR = os.path.join(BASE_DIR, "weights")
OUTPUT_MODEL_PATH = os.path.join(OUTPUT_DIR, "yolop-640-640-quantized-int8-INC.onnx")
CALIBRATION_IMG_DIR = os.path.join(BASE_DIR, "dataset", "images", "val")
CALIBRATION_SIZE = 100
TARGET_SIZE = (640, 640)


In [4]:


class YOLOPDataReader(CalibrationDataReader):
    """DataReader avec preprocessing YOLOP standard"""
    
    def __init__(self, image_folder, model_path, size_limit=100):
        self.image_folder = image_folder
        self.enum_data_dicts = None
        self.datasize = 0
        
        session = onnxruntime.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        self.input_name = session.get_inputs()[0].name
        
        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        files = [f for f in os.listdir(image_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]
        self.image_files = files[:size_limit]
        self.datasize = len(self.image_files)
        print(f" DataReader : {self.datasize} images de calibration")
    
    def letterbox_image(self, image, size):

        iw, ih = image.size
        w, h = size
        scale = min(w/iw, h/ih)
        nw = int(iw * scale)
        nh = int(ih * scale)
        
        image = image.resize((nw, nh), Image.BICUBIC)
        new_image = Image.new('RGB', size, (114, 114, 114))
        new_image.paste(image, ((w - nw) // 2, (h - nh) // 2))

        return new_image
    
    def get_next(self):
        if self.enum_data_dicts is None:
            self.enum_data_dicts = iter(self.load_batch())
        try:
            return next(self.enum_data_dicts)
        except StopIteration:
            return None
    
    def load_batch(self):
        for filename in self.image_files:
            image_path = os.path.join(self.image_folder, filename)
            image = Image.open(image_path).convert("RGB")
            image = self.letterbox_image(image, TARGET_SIZE)
            
            input_tensor = self.normalize(image)
            input_numpy = input_tensor.unsqueeze(0).numpy()
            
            yield {self.input_name: input_numpy}



In [5]:
class INC_YOLOPDataset:
    """Dataset compatible Intel Neural Compressor"""
    def __init__(self, image_folder, size_limit=50):
        self.files = [os.path.join(image_folder, f) for f in os.listdir(image_folder) if f.endswith('.jpg')][:size_limit]
        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        
        img = Image.open(self.files[index]).convert("RGB")
        iw, ih = img.size
        w, h = 640, 640 
        scale = min(w/iw, h/ih)
        nw, nh = int(iw*scale), int(ih*scale)
        img_resized = img.resize((nw,nh), Image.BICUBIC)
        new_image = Image.new('RGB', (w, h), (114,114,114))
        new_image.paste(img_resized, ((w-nw)//2, (h-nh)//2))
        
        input_tensor = self.normalize(new_image)
        return input_tensor.numpy(), 0

In [6]:

def get_bn_weight_names(model_path):
    """Identifie tous les noms de poids BatchNorm dans le modèle"""
    model = onnx.load(model_path)
    bn_weights = []
    
    for initializer in model.graph.initializer:
        if '.bn.weight' in initializer.name or '.bn.bias' in initializer.name:
            bn_weights.append(initializer.name)
    
    print(f" {len(bn_weights)} poids BatchNorm identifiés")
    return bn_weights



In [7]:
def get_sensitive_nodes(model_path, output_name='det_out', depth=3):
  
    model = onnx.load(model_path)
    nodes_to_exclude = []
    
    output_map = {}
    for node in model.graph.node:
        for out in node.output:
            output_map[out] = node.name
            
    current_nodes = []
    for node in model.graph.node:
        if output_name in node.output:
            current_nodes.append(node)
            nodes_to_exclude.append(node.name)
    
    print(f" Sortie '{output_name}' produite par : {[n.name for n in current_nodes]}")

    for i in range(depth):
        parents = []
        for node in current_nodes:
            for inp in node.input:
                if inp in output_map:
                    parent_name = output_map[inp]
                    nodes_to_exclude.append(parent_name)
                    
                    for n in model.graph.node:
                        if n.name == parent_name:
                            parents.append(n)
                            break
        current_nodes = parents
    
    nodes_to_exclude = list(set(nodes_to_exclude))
    print(f"   {len(nodes_to_exclude)} noeuds sensibles identifiés pour exclusion (Mixed Precision)")
    return nodes_to_exclude

In [8]:
# --- METRIQUE DE SURVIE POUR INC ---
class ConfidenceSurvivalMetric(Metric):
    def __init__(self, name="conf_check"):
        self.name = name
        self.scores = []

    def update(self, preds, labels):
        
        try:
            det_out = preds[0] if isinstance(preds, list) else preds
            
            max_conf = np.max(det_out[:, :, 4])
            self.scores.append(max_conf)
        except:
            self.scores.append(0.0)

    def reset(self):
        self.scores = []

    def result(self):
        avg = np.mean(self.scores)
        print(f"   [Métrique INC] Confiance Moyenne : {avg:.5f}")
        return avg

In [9]:

def quantize_model():
    print("=" * 70)
    print("  QUANTIFICATION YOLOP POUR DÉPLOIEMENT NPU (INT8)")
    print("=" * 70)
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    if not os.path.exists(INPUT_MODEL_PATH):
        print(f" ERREUR : Modèle introuvable : {INPUT_MODEL_PATH}")
        return
    
    # ============================================================
    # ÉTAPE 1 : SIMPLIFICATION (Fusion Conv+BN)
    # ============================================================
    
    print("\n[1/4] Simplification du modèle ")
    SIMPLIFIED_PATH = os.path.join(OUTPUT_DIR, "temp_simplified.onnx")
    
    model_proto = onnx.load(INPUT_MODEL_PATH)
    model_sim, check = simplify(model_proto)
    
    if not check:
        print(" Échec de la simplification")
        return
    
    onnx.save(model_sim, SIMPLIFIED_PATH)
    print(f" Modèle simplifié : {SIMPLIFIED_PATH}")

    print(f"    Validation et pré-traitement ONNX Runtime...")
    PREPROCESSED_PATH = os.path.join(OUTPUT_DIR, "temp_preprocessed.onnx")
    quant_pre_process(SIMPLIFIED_PATH, PREPROCESSED_PATH)
    
    if os.path.exists(SIMPLIFIED_PATH):
        os.remove(SIMPLIFIED_PATH)
        
    SIMPLIFIED_PATH = PREPROCESSED_PATH
    print(f" Modèle pré-traité : {SIMPLIFIED_PATH}")

    # ============================================================
    # ÉTAPE 2 : TUNING AUTOMATIQUE (INTEL NEURAL COMPRESSOR)
    # ============================================================
    print("\n[2/2] Lancement du Tuning Automatique INC...")
    print("      Objectif : Trouver le mix INT8/FP32 qui garde la détection active.")
    
    dataset = INC_YOLOPDataset(CALIBRATION_IMG_DIR, size_limit=CALIBRATION_SIZE)
    dataloader = DataLoader(framework='onnxruntime', dataset=dataset, batch_size=1)
    
    def eval_func(model):
        session = onnxruntime.InferenceSession(model.SerializeToString(), providers=['CPUExecutionProvider'])
        input_name = session.get_inputs()[0].name
        
        total_conf = 0.0
        count = 0
        
        for inputs, _ in dataloader:
            preds = session.run(None, {input_name: inputs})
            
            det_out = preds[0]
            batch_max_conf = np.max(det_out[:, :, 4])
            
            total_conf += batch_max_conf
            count += 1
            
        avg_conf = total_conf / count if count > 0 else 0.0
        print(f"   [Eval] Confiance Moyenne : {avg_conf:.5f}")
        return avg_conf
    
    accuracy_crit = AccuracyCriterion(higher_is_better=True, criterion='relative', tolerable_loss=0.1)
    tuning_crit = TuningCriterion(max_trials=15, timeout=0)

    config = PostTrainingQuantConfig(
        approach='static',
        tuning_criterion=tuning_crit,
        accuracy_criterion=accuracy_crit
    )
    
    try:
        q_model = quantization.fit(
            model=SIMPLIFIED_PATH,
            conf=config,
            calib_dataloader=dataloader, 
            eval_func=eval_func         
        )
        
        if q_model:
            print("\n" + "=" * 70)
            print("  SUCCÈS : INC A TROUVÉ UN MODÈLE FONCTIONNEL ")
            print("=" * 70)
            q_model.save(OUTPUT_MODEL_PATH)
            
            print(f" Modèle sauvegardé : {OUTPUT_MODEL_PATH}")
            
            src = os.path.join(OUTPUT_DIR, "best_model.onnx")
            if os.path.exists(src):
                shutil.move(src, OUTPUT_MODEL_PATH)
                print(f" Modèle sauvegardé : {OUTPUT_MODEL_PATH}")
                
                if os.path.exists(SIMPLIFIED_PATH): os.remove(SIMPLIFIED_PATH)
            else:
                print(f" Le modèle est dans {OUTPUT_DIR} (probablement 'best_model.onnx')")
        else:
            print(" ÉCHEC : INC n'a pas réussi à stabiliser le modèle.")
            
    except Exception as e:
        print(f"\n ERREUR INC : {str(e)}")
        import traceback
        traceback.print_exc()

In [10]:

if __name__ == "__main__":
    quantize_model()

  QUANTIFICATION YOLOP POUR DÉPLOIEMENT NPU (INT8)

[1/4] Simplification du modèle 
 Modèle simplifié : /workspace/weights/temp_simplified.onnx
    Validation et pré-traitement ONNX Runtime...


2026-02-04 07:54:18 [INFO] Start auto tuning.
2026-02-04 07:54:18 [INFO] Execute the tuning process due to detect the evaluation function.
2026-02-04 07:54:18 [INFO] Adaptor has 5 recipes.
2026-02-04 07:54:18 [INFO] 0 recipes specified by user.
2026-02-04 07:54:18 [INFO] 3 recipes require future tuning.
2026-02-04 07:54:18 [INFO] *** Initialize auto tuning
2026-02-04 07:54:18 [INFO] {
2026-02-04 07:54:18 [INFO]     'PostTrainingQuantConfig': {
2026-02-04 07:54:18 [INFO]         'AccuracyCriterion': {
2026-02-04 07:54:18 [INFO]             'criterion': 'relative',
2026-02-04 07:54:18 [INFO]             'higher_is_better': True,
2026-02-04 07:54:18 [INFO]             'tolerable_loss': 0.1,
2026-02-04 07:54:18 [INFO]             'absolute': None,
2026-02-04 07:54:18 [INFO]             'keys': <bound method AccuracyCriterion.keys of <neural_compressor.config.AccuracyCriterion object at 0x760e4368c0a0>>,
2026-02-04 07:54:18 [INFO]             'relative': 0.1
2026-02-04 07:54:18 [INFO]      

 Modèle pré-traité : /workspace/weights/temp_preprocessed.onnx

[2/2] Lancement du Tuning Automatique INC...
      Objectif : Trouver le mix INT8/FP32 qui garde la détection active.


2026-02-04 07:54:18 [WARNING] This model has nodes with the same name, please checkrenamed_model.onnx in workspace_path (default is nc_workspace)for newly generated node name
2026-02-04 07:54:19 [INFO] Get FP32 model baseline.
2026-02-04 07:54:29 [INFO] Save tuning history to /workspace/nc_workspace/2026-02-04_07-54-14/./history.snapshot.


   [Eval] Confiance Moyenne : 0.90786


2026-02-04 07:54:29 [INFO] FP32 baseline is: [Accuracy: 0.9079, Duration (seconds): 9.6295]
2026-02-04 07:54:29 [INFO] Quantize the model with default config.
2026-02-04 07:56:56 [INFO] |*******Mixed Precision Statistics*******|
2026-02-04 07:56:56 [INFO] +------------------+-------+------+------+
2026-02-04 07:56:56 [INFO] |     Op Type      | Total | INT8 | FP32 |
2026-02-04 07:56:56 [INFO] +------------------+-------+------+------+
2026-02-04 07:56:56 [INFO] |       Conv       |  102  | 102  |  0   |
2026-02-04 07:56:56 [INFO] |       Mul        |   87  |  78  |  9   |
2026-02-04 07:56:56 [INFO] |    LeakyRelu     |   12  |  12  |  0   |
2026-02-04 07:56:56 [INFO] |     Sigmoid      |   5   |  5   |  0   |
2026-02-04 07:56:56 [INFO] |     MaxPool      |   3   |  0   |  3   |
2026-02-04 07:56:56 [INFO] |       Add        |   85  |  75  |  10  |
2026-02-04 07:56:56 [INFO] |     Reshape      |   15  |  6   |  9   |
2026-02-04 07:56:56 [INFO] |      Concat      |   19  |  12  |  7   |
2

   [Eval] Confiance Moyenne : 0.90271

  SUCCÈS : INC A TROUVÉ UN MODÈLE FONCTIONNEL 
 Modèle sauvegardé : /workspace/weights/yolop-640-640-quantized-int8-INC.onnx
 Le modèle est dans /workspace/weights (probablement 'best_model.onnx')
